# 02 - Training: Legacy SplatFields Pipeline

This notebook handles training of the SplatFields model using the prepared dataset.

**Runtime Environment:** Legacy/Custom (PyTorch 1.12, CUDA 11.6)

**CRITICAL NOTES:**
- PyTorch 1.12 is **incompatible** with H100 (Hopper) GPUs - use T4, L4, or A100
- Data is copied from Drive to local disk to prevent I/O bottleneck
- All installation commands use verbose output for troubleshooting

**IMPORTANT:** Do NOT use H100 GPUs with this notebook!

## 1. Setup & Drive Mount

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define project paths - CUSTOMIZE THESE TO MATCH YOUR DATA PREP NOTEBOOK
PROJECT_ROOT = "/content/drive/MyDrive/SplatFields_Project"
DRIVE_DATASET_PATH = f"{PROJECT_ROOT}/ready_to_train_dataset"

# Local paths (for training - MANDATORY for performance)
LOCAL_DATASET_PATH = "/content/dataset"
OUTPUT_PATH = "/content/output"

# Repository URL (the refactored SplatFields repository)
REPO_URL = "https://github.com/semhfe/SplatFields-FE.git"  # Update with your repo URL
REPO_BRANCH = "copilot/refactor-4d-reconstruction-pipeline"  # Branch with the refactored code

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DRIVE_DATASET_PATH: {DRIVE_DATASET_PATH}")
print(f"LOCAL_DATASET_PATH: {LOCAL_DATASET_PATH}")

## 2. Hardware Check & Environment Configuration

Check GPU compatibility and set appropriate CUDA architecture flags.

In [ ]:
import subprocess
import os

print("="*60)
print("GPU HARDWARE CHECK")
print("="*60)

# Run nvidia-smi
!nvidia-smi

# Get GPU name
result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], 
                       capture_output=True, text=True)
gpu_name = result.stdout.strip()
print(f"\nDetected GPU: {gpu_name}")

In [ ]:
# Architecture check and configuration
import sys

# Check for H100 (Hopper architecture) - INCOMPATIBLE with PyTorch 1.12
if 'H100' in gpu_name or 'Hopper' in gpu_name.lower():
    print("\n" + "!"*60)
    print("⚠️  CRITICAL WARNING: H100 (Hopper) GPU DETECTED!")
    print("!"*60)
    print("\nPyTorch 1.12 with CUDA 11.6 is INCOMPATIBLE with H100 GPUs.")
    print("H100 requires CUDA 11.8+ and PyTorch 2.0+.")
    print("\nPLEASE SWITCH TO A DIFFERENT GPU:")
    print("  - T4 (Recommended)")
    print("  - L4")
    print("  - A100")
    print("\nTo change GPU: Runtime -> Change runtime type -> Hardware accelerator")
    print("!"*60)
    
    # Raise error to prevent continuing
    raise RuntimeError("H100 GPU is incompatible with this training pipeline. Please use T4, L4, or A100.")

# Set CUDA architecture flags based on GPU
if 'A100' in gpu_name:
    os.environ['TORCH_CUDA_ARCH_LIST'] = '8.0'
    print(f"\n✓ A100 detected. Setting TORCH_CUDA_ARCH_LIST=8.0")
elif 'L4' in gpu_name:
    os.environ['TORCH_CUDA_ARCH_LIST'] = '8.9'
    print(f"\n✓ L4 detected. Setting TORCH_CUDA_ARCH_LIST=8.9")
elif 'T4' in gpu_name:
    os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
    print(f"\n✓ T4 detected. Setting TORCH_CUDA_ARCH_LIST=7.5")
elif 'V100' in gpu_name:
    os.environ['TORCH_CUDA_ARCH_LIST'] = '7.0'
    print(f"\n✓ V100 detected. Setting TORCH_CUDA_ARCH_LIST=7.0")
else:
    # Default: support multiple architectures
    os.environ['TORCH_CUDA_ARCH_LIST'] = '7.0;7.5;8.0;8.6;8.9'
    print(f"\n⚠️  Unknown GPU: {gpu_name}")
    print(f"Setting TORCH_CUDA_ARCH_LIST=7.0;7.5;8.0;8.6;8.9")

print(f"\nTORCH_CUDA_ARCH_LIST = {os.environ.get('TORCH_CUDA_ARCH_LIST')}")
print("\n✓ Hardware check passed!")

## 3. Install Legacy Environment

Install PyTorch 1.12.1 with CUDA 11.6 and required dependencies.

In [ ]:
# Uninstall default PyTorch (Colab comes with PyTorch 2.x)
print("="*60)
print("INSTALLING LEGACY PYTORCH ENVIRONMENT")
print("="*60)

print("\n[1/5] Uninstalling default PyTorch...")
!pip uninstall torch torchvision torchaudio -y

print("\n✓ Default PyTorch uninstalled")

In [ ]:
# Install PyTorch 1.12.1 with CUDA 11.6
print("\n[2/5] Installing PyTorch 1.12.1 + CUDA 11.6...")
!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 --extra-index-url https://download.pytorch.org/whl/cu116

print("\n✓ PyTorch 1.12.1 installed")

In [ ]:
# Verify PyTorch installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Verify correct version
assert torch.__version__.startswith('1.12'), f"Expected PyTorch 1.12.x, got {torch.__version__}"
print("\n✓ PyTorch verification passed!")

In [ ]:
# Install CUDA toolkit 11.6
print("\n[3/5] Installing CUDA toolkit 11.6...")
!apt-get update
!apt-get install -y cuda-toolkit-11-6

# Set CUDA paths
os.environ['CUDA_HOME'] = '/usr/local/cuda-11.6'
os.environ['PATH'] = f"/usr/local/cuda-11.6/bin:{os.environ.get('PATH', '')}"
os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda-11.6/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

print(f"\nCUDA_HOME: {os.environ.get('CUDA_HOME')}")
!nvcc --version

In [ ]:
# Clone the SplatFields repository
print("\n[4/5] Cloning SplatFields repository...")

REPO_DIR = "/content/SplatFields"

# Remove if exists
!rm -rf {REPO_DIR}

# Clone with the refactored branch
!git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}

# Change to repo directory
%cd {REPO_DIR}

print(f"\n✓ Repository cloned to {REPO_DIR}")
!git log --oneline -3

In [ ]:
# Install diff-gaussian-rasterization (compile from source with verbose output)
print("\n[5a/5] Installing diff-gaussian-rasterization...")

# Clone submodule if needed
!git submodule update --init --recursive

# Check if submodule exists, otherwise clone manually
DIFF_RAST_DIR = f"{REPO_DIR}/submodules/diff-gaussian-rasterization"
if not os.path.exists(DIFF_RAST_DIR):
    print("Cloning diff-gaussian-rasterization...")
    !git clone https://github.com/graphdeco-inria/diff-gaussian-rasterization.git {DIFF_RAST_DIR}

%cd {DIFF_RAST_DIR}
!pip install . --verbose

print("\n✓ diff-gaussian-rasterization installed")

In [ ]:
# Install simple-knn (compile from source with verbose output)
print("\n[5b/5] Installing simple-knn...")

SIMPLE_KNN_DIR = f"{REPO_DIR}/submodules/simple-knn"
if not os.path.exists(SIMPLE_KNN_DIR):
    print("Cloning simple-knn...")
    !git clone https://gitlab.inria.fr/bkerbl/simple-knn.git {SIMPLE_KNN_DIR}

%cd {SIMPLE_KNN_DIR}
!pip install . --verbose

print("\n✓ simple-knn installed")

In [ ]:
# Install remaining dependencies
print("\n[5c/5] Installing remaining dependencies...")

%cd {REPO_DIR}
!pip install plyfile tqdm opencv-python-headless Pillow trimesh scikit-learn imageio

print("\n" + "="*60)
print("LEGACY ENVIRONMENT INSTALLATION COMPLETE")
print("="*60)

## 4. Data Loading (Copy from Drive to Local)

**MANDATORY:** Copy the dataset from Drive to local disk to prevent I/O bottleneck during training.

In [ ]:
import shutil
import json
import os

print("="*60)
print("COPYING DATASET TO LOCAL DISK")
print("="*60)
print("\n⚠️  This is MANDATORY to prevent I/O latency during training!")
print(f"\nSource (Drive): {DRIVE_DATASET_PATH}")
print(f"Destination (Local): {LOCAL_DATASET_PATH}")

# Verify source exists
if not os.path.exists(DRIVE_DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found at {DRIVE_DATASET_PATH}. Please run the data prep notebook first.")

# Remove existing local dataset
if os.path.exists(LOCAL_DATASET_PATH):
    print("\nRemoving existing local dataset...")
    shutil.rmtree(LOCAL_DATASET_PATH)

# Copy dataset
print("\nCopying dataset (this may take a few minutes)...")
shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)

# Verify copy
local_files = []
for root, dirs, files in os.walk(LOCAL_DATASET_PATH):
    local_files.extend(files)
    
print(f"\n✓ Copied {len(local_files)} files to local disk")

# List structure
print("\nDataset structure:")
!ls -la {LOCAL_DATASET_PATH}

In [ ]:
# Read metadata.json to get camera names
metadata_path = os.path.join(LOCAL_DATASET_PATH, 'metadata.json')

if os.path.exists(metadata_path):
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    train_cam_names = metadata.get('train_cam_names', [])
    print(f"Loaded metadata:")
    print(f"  - Number of cameras: {len(train_cam_names)}")
    print(f"  - Camera names: {train_cam_names}")
else:
    print(f"⚠️  metadata.json not found at {metadata_path}")
    train_cam_names = []

# Convert camera names list to space-separated string for command line
train_cam_names_str = ' '.join(train_cam_names)
print(f"\nCamera names for training: {train_cam_names_str}")

## 5. Training

Run the SplatFields training with the prepared dataset.

In [ ]:
# Create output directory
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Change to repository directory
%cd {REPO_DIR}

print("="*60)
print("TRAINING CONFIGURATION")
print("="*60)
print(f"\nSource path: {LOCAL_DATASET_PATH}")
print(f"Point cloud: {LOCAL_DATASET_PATH}/init_ply/dense.ply")
print(f"Output path: {OUTPUT_PATH}")
print(f"Training cameras: {train_cam_names}")
print(f"White background: True")
print(f"Lambda mask: 0.1")

In [ ]:
# Construct and run the training command
import subprocess

# Build training command
train_cmd = [
    'python', 'train.py',
    '--source_path', LOCAL_DATASET_PATH,
    '--model_path', OUTPUT_PATH,
    '--pc_path', f'{LOCAL_DATASET_PATH}/init_ply/dense.ply',
    '--white_background',
    '--lambda_mask', '0.1',
    '--eval',
]

# Add camera names if available
if train_cam_names:
    train_cmd.extend(['--train_cam_names'] + train_cam_names)

print("Training command:")
print(' '.join(train_cmd))
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60 + "\n")

In [ ]:
# Run training
# Using ! for better real-time output in Colab

pc_path = f"{LOCAL_DATASET_PATH}/init_ply/dense.ply"

# Format camera names for command line
if train_cam_names:
    cam_names_arg = f"--train_cam_names {' '.join(train_cam_names)}"
else:
    cam_names_arg = ""

!python train.py \
    --source_path {LOCAL_DATASET_PATH} \
    --model_path {OUTPUT_PATH} \
    --pc_path {pc_path} \
    --white_background \
    --lambda_mask 0.1 \
    --eval \
    {cam_names_arg}

## 6. Render & Export

In [ ]:
# Run rendering
print("="*60)
print("RENDERING")
print("="*60)

!python render.py \
    --source_path {LOCAL_DATASET_PATH} \
    --model_path {OUTPUT_PATH}

print("\n✓ Rendering complete")

In [ ]:
# Persist output to Google Drive
import shutil
from datetime import datetime

print("="*60)
print("SAVING OUTPUT TO GOOGLE DRIVE")
print("="*60)

# Create timestamp for output folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_output_path = f"{PROJECT_ROOT}/training_output_{timestamp}"

print(f"\nCompressing output folder...")
zip_path = f"/content/output_{timestamp}.zip"
!cd /content && zip -r {zip_path} output/

print(f"\nCopying to Drive: {drive_output_path}")
os.makedirs(drive_output_path, exist_ok=True)

# Copy zip to drive
shutil.copy2(zip_path, f"{drive_output_path}/output.zip")

# Also copy some key files directly
if os.path.exists(f"{OUTPUT_PATH}/cfg_args"):
    shutil.copy2(f"{OUTPUT_PATH}/cfg_args", f"{drive_output_path}/cfg_args")

print(f"\n✓ Output saved to {drive_output_path}")
print(f"\nContents:")
!ls -la {drive_output_path}

## 7. Summary

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\nOutput saved to:")
print(f"  Local: {OUTPUT_PATH}")
print(f"  Drive: {drive_output_path}")
print("\nOutput contents:")
!ls -la {OUTPUT_PATH}

print("\n" + "="*60)
print("\nTo use the trained model:")
print(f"1. Download from Drive: {drive_output_path}/output.zip")
print("2. Extract and use with render.py or your visualization tool")
print("="*60)